# InvisiPhish - semantic classifier training

Trains the DistilBERT phishing classifier described in the paper and exports it to ONNX so the **same weights** run in the browser (`@xenova/transformers`) and in the Node evaluation harness (`npm run eval`).

The paper's configuration is used unchanged: AdamW, learning rate 3e-4, batch size 16, up to 8 epochs, cross-entropy loss, early stopping on validation F1, metrics accuracy / precision / recall / F1 / ROC-AUC.

**Before running:** add a labelled phishing dataset as an input to this notebook, and make the `invisiphish` repository public (or upload it as a dataset) so cell 3 can fetch it.

In [ ]:
!pip install -q --upgrade accelerate transformers onnxruntime onnx scikit-learn numpy pandas

import torch, transformers, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
print('transformers', transformers.__version__)

## 1. Configuration

- `REPO_URL` - the GitHub URL of your `invisiphish` repository (make it public, or switch to the *upload repo as dataset* method below).
- `DATASET_PATH` - where your labelled corpus is mounted. The script auto-discovers the largest usable CSV, or a directory whose sub-folder names encode the label (`phishing_emails/`, `legitimate_emails/`).
- `MODEL` - `distilbert-base-uncased` for the shipped model. `bert-base-uncased` or `microsoft/deberta-v3-base` give stronger upper bounds for the paper's comparison table.

In [ ]:
import os, glob

REPO_URL      = 'https://github.com/YOUR_USERNAME/invisiphish.git'
REPO_DIR      = '/kaggle/working/invisiphish'
DATASET_PATH  = '/kaggle/input'          # narrow this to your dataset folder
MODEL         = 'distilbert-base-uncased'
SEEDS         = '13,42,77'              # training-variance across seeds
EPOCHS        = 8
BATCH_SIZE    = 16
OUTPUT_DIR    = '/kaggle/working/models'

print('available inputs:')
for path in sorted(glob.glob('/kaggle/input/*')):
    print('  ', path)


## 2. Fetch the repository

If you uploaded the repository as a Kaggle dataset instead of cloning it, set `REPO_DIR` to that path and skip this cell.

In [ ]:
import subprocess

subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
print('cloned to', REPO_DIR)
print(subprocess.run(['ls', REPO_DIR], capture_output=True, text=True).stdout)

## 3. Inspect the discovered dataset

The loader prints the file it chose, the message count and the class balance. Check that these look right before spending GPU time.

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'ml'))
from pathlib import Path
from train_distilbert import discover_dataset

rows, description = discover_dataset(Path(DATASET_PATH))
pos = sum(1 for label, _ in rows if label == 'phishing')
print('source      :', description)
print('messages    :', len(rows))
print('phishing    :', pos, f'({pos / max(1, len(rows)):.1%})')
print('sample      :', rows[0][1][:160] if rows else 'none')


In [ ]:
## 4. Train

## 4. Train

Each seed writes its own exported model to `$OUTPUT_DIR/seed-<n>/`, together with a `training_report.json` and a quantised ONNX copy for browser deployment.

In [ ]:
command = [
    'python', os.path.join(REPO_DIR, 'ml', 'train_distilbert.py'),
    '--data', DATASET_PATH,
    '--output-dir', OUTPUT_DIR,
    '--model', MODEL,
    '--seeds', SEEDS,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--copy-primary-to', os.path.join(OUTPUT_DIR, 'primary'),
]
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import json

summary = json.load(open(os.path.join(OUTPUT_DIR, 'training_summary.json')))
print('model  :', summary['base_model'])
print('dataset:', summary['dataset'])
print()
print(f"{'metric':<12}{'mean':>10}{'std':>10}   per-seed")
for key, stats in summary['summary'].items():
    print(f"{key:<12}{stats['mean']:>10.4f}{stats['std']:>10.4f}   {stats['values']}")


## 5. Package the artifacts

Download `invisiphish-model.zip` from the notebook output and unpack it into `public/models/` in your local checkout:

```bash
unzip invisiphish-model.zip -d public/models/
# -> public/models/distilbert-phishing/   (config.json, tokenizer files, onnx/model.onnx)
```

Then run the evaluation harness locally, which uses the exported model and the shipped TypeScript pipeline:

```bash
npm run eval -- --seeds 13,42,77
```

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/invisiphish-model', 'zip', OUTPUT_DIR)
print('wrote /kaggle/working/invisiphish-model.zip')
